In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr

@dp.view
def gold_shipment_changes():
    return (
        spark.readStream
        .option("readChangeFeed", "true")
        .table("shipments_silver")
        .filter(col("_change_type").isin(
            "insert",
            "update_postimage",
            "delete"
        ))
        .select(
            col("shipment_id").alias("ShipmentId"),
            col("status").alias("Status"),
            col("location").alias("Location"),
            col("customer_name").alias("CustomerName"),
            col("_change_type"),
            col("_commit_version")
        )
    )


dp.create_streaming_table("shipments_gold")

dp.create_auto_cdc_flow(
    target="shipments_gold",
    source="gold_shipment_changes",
    keys=["ShipmentId"],
    sequence_by=col("_commit_version"),
    apply_as_deletes=expr("_change_type = 'delete'"),
    except_column_list=[
        "_change_type",
        "_commit_version"
    ],
    stored_as_scd_type="1"
)